In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import time
from datetime import datetime
from scipy import stats

In [ ]:
# 设置全局字体为黑体
plt.rcParams['font.family'] = 'SimHei'
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
#读取附件中的男胎数据
male_data = pd.read_excel('附件.xlsx', sheet_name='男胎检测数据')

In [ ]:
male_data.isna().sum()

In [ ]:
#删去末次月经为空的数据
male_data = male_data.dropna(subset=['末次月经'])
#仅保留自然受孕的，删去人工授精
male_data = male_data[male_data['IVF妊娠']=='自然受孕']

In [ ]:
plt.figure(figsize=(8,7))
sns.boxplot(male_data[['13号染色体的Z值','18号染色体的Z值','21号染色体的Z值','X染色体的Z值','Y染色体的Z值']])

In [ ]:
sns.boxplot(male_data['孕妇BMI'])

In [ ]:
sns.boxplot(male_data['Y染色体浓度'])

In [ ]:
#根据3sigma原则删去偏离值
male_data = male_data[(-3<male_data['13号染色体的Z值']) & (male_data['13号染色体的Z值']<3) ]
male_data = male_data[(-3<male_data['18号染色体的Z值']) & (male_data['18号染色体的Z值']<3) ]
male_data = male_data[(-3<male_data['21号染色体的Z值']) & (male_data['21号染色体的Z值']<3) ]
male_data = male_data[(-3<male_data['X染色体的Z值']) & (male_data['X染色体的Z值']<3) ]
male_data = male_data[(-3<male_data['Y染色体的Z值']) & (male_data['Y染色体的Z值']<3) ]

In [ ]:
#根据1.5IQR原则删去BMI异常值
BMI_Q1 = male_data['孕妇BMI'].quantile(0.25)
BMI_Q3 = male_data['孕妇BMI'].quantile(0.75)
BMI_IQR = BMI_Q3-BMI_Q1
BMI_lower = BMI_Q1 - 1.5 * BMI_IQR
BMI_upper = BMI_Q3 + 1.5 * BMI_IQR
male_data = male_data[(BMI_lower<male_data['孕妇BMI']) & (male_data['孕妇BMI']<BMI_upper) ]

In [ ]:
#根据1.5IQR原则删去Y染色体浓度异常值
Y_Q1 = male_data['Y染色体浓度'].quantile(0.25)
Y_Q3 = male_data['Y染色体浓度'].quantile(0.75)
Y_IQR = Y_Q3-Y_Q1
Y_lower = Y_Q1 - 1.5 * Y_IQR
Y_upper = Y_Q3 + 1.5 * Y_IQR
male_data = male_data[(Y_lower<male_data['Y染色体浓度']) & (male_data['Y染色体浓度']<Y_upper) ]

In [ ]:
#序号重新编排
male_data = male_data.reset_index()

In [ ]:
#将孕周格式统一为一个小数
def transform_yunzhou(data):
    i = 0
    weeks = 0
    while data[i]!='w' and data[i]!='W':
        i+=1
    weeks += int(data[:i])
    if data[-1]=='w' or data[-1]=='W':
        return weeks
    if data[i+1]=='+':
        weeks += int(data[i+2])/7
    return weeks

In [ ]:
male_data['检测孕周'] = male_data['检测孕周'].apply(lambda x:transform_yunzhou(x))

In [ ]:
male_data

In [ ]:
#取出数据中数值型的列
mvalue_data = male_data[['年龄','身高','体重','检测孕周','孕妇BMI','原始读段数','在参考基因组上比对的比例','重复读段的比例','唯一比对的读段数  ',
                        'GC含量','13号染色体的Z值','18号染色体的Z值','21号染色体的Z值','X染色体的Z值','Y染色体的Z值','Y染色体浓度',
                         'X染色体浓度','13号染色体的GC含量','18号染色体的GC含量','21号染色体的GC含量','被过滤掉读段数的比例']]

In [ ]:
#KS正态性检验
ks_male = pd.DataFrame(columns=['项目','p-value'])
for col in mvalue_data.columns:
    ks_male.loc[len(ks_male)] = [col,stats.kstest(mvalue_data[col],'norm')[1]]

In [ ]:
ks_male

In [ ]:
#计算相关系数
corr = mvalue_data.corr(method='spearman')

In [ ]:
#画出相关系数热力图
plt.figure(figsize=(15,10))
sns.heatmap(corr,annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.8)

In [ ]:
#计算与Y染色体浓度显著性P-value
spearman_male = pd.DataFrame(columns=['项目','p-value'])
for col in mvalue_data.columns:
    spearman_male.loc[len(spearman_male)] = [col,stats.spearmanr(mvalue_data['Y染色体浓度'],mvalue_data[col])[1]]

In [ ]:
spearman_male

In [ ]:
spearman_male['相关系数'] = corr['Y染色体浓度'].values

In [ ]:
mvalue_data.to_csv('male.csv')

In [ ]:
male_data.to_csv('male_data.csv')

In [ ]:
plt.scatter(mvalue_data['检测孕周'],mvalue_data['Y染色体浓度'])

In [ ]:
male_data

In [ ]:
md=male_data[['孕妇代码','检测孕周','孕妇BMI','Y染色体浓度']]
md.to_csv('md.csv')

In [ ]:
import numpy as np

# 模型拟合
model = smf.mixedlm("Q('Y染色体浓度') ~ Q('检测孕周') + Q('孕妇BMI')",
                    data=male_data,
                    groups=male_data["孕妇代码"])
result = model.fit()

# 固定效应预测值
y_fixed = result.fittedvalues

# 总方差
var_y = np.var(male_data["Y染色体浓度"])

# 固定效应解释的方差
var_fixed = np.var(y_fixed)

# 随机效应和残差的方差
var_random = float(result.cov_re.iloc[0,0])   # 随机截距方差
var_resid  = result.scale                     # 残差方差

# Marginal R²（固定效应）
R2_marginal = var_fixed / (var_fixed + var_random + var_resid)

# Conditional R²（固定+随机效应）
R2_conditional = (var_fixed + var_random) / (var_fixed + var_random + var_resid)

print("Marginal R²:", R2_marginal)
print("Conditional R²:", R2_conditional)
